In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report)
import json, os

df = pd.read_csv('../data/raw/rice.csv')
print(df.shape)
df.head()


(3810, 8)


,Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class
0,15231.0,525.578979,229.749878,85.093788,0.928882,15617.0,0.572896,Cammeo
1,14656.0,494.311005,206.020065,91.730972,0.895405,15072.0,0.615436,Cammeo
2,14634.0,501.122009,214.106781,87.768288,0.912118,14954.0,0.693259,Cammeo
3,13176.0,458.342987,193.337387,87.448395,0.891861,13368.0,0.640669,Cammeo
4,14688.0,507.166992,211.743378,89.312454,0.906691,15262.0,0.646024,Cammeo


In [2]:

df['Class_encoded'] = df['Class'].astype('category').cat.codes


In [3]:
# Train/test split (same fixed split used by every member's model, for fair comparison)
X = df.drop(columns=['Class_encoded']).values
y = df['Class_encoded'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Further split a validation set out of the training data (for Keras' validation_data)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)
print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)


Train: (2590, 8)  Val: (458, 8)  Test: (762, 8)


In [9]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# NumPy array/List ekak nam Pandas DataFrame ekakata harawaganna
if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train)

# Features (X) wala text columns encoding
X_train = pd.get_dummies(X_train, drop_first=True)

# Target variable (y) eka text nam encode karanna
le = LabelEncoder()
y_train = le.fit_transform(y_train)

In [10]:
#Hyperparameter tuning with GridSearchCV (5-fold)
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5,
                     scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
best_params = grid.best_params_
print("Best params:", best_params)


Best params: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train, y_train, cv=cv, scoring='accuracy')
print("CV accuracy per fold:", cv_scores.round(4))
print(f"CV mean accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")


CV accuracy per fold: [1. 1. 1. 1. 1.]
CV mean accuracy: 1.0000 (+/- 0.0000)


In [14]:
import pandas as pd

# X_test DataFrame ekak nemei nam DataFrame ekakata harawanna
if not isinstance(X_test, pd.DataFrame):
    X_test = pd.DataFrame(X_test)

# X_test text columns encoding (X_train ekata kalama wagema)
X_test = pd.get_dummies(X_test, drop_first=True)

# y_test text values encode karanna
y_test = le.transform(y_test)

In [15]:
pos_val = 1 

# Generate predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate classification performance metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=pos_val)
rec = recall_score(y_test, y_pred, pos_label=pos_val)
f1 = f1_score(y_test, y_pred, pos_label=pos_val)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4.5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cammeo','Osmancik'], yticklabels=['Cammeo','Osmancik'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.savefig('../results/eda_visualizations/random_forest_confusion_matrix.png', dpi=150)
plt.show()


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- 0_10020.0
- 0_10023.0
- 0_10025.0
- 0_10083.0
- 0_10106.0
- ...
Feature names seen at fit time, yet now missing:
- 0_10000.0
- 0_10005.0
- 0_10007.0
- 0_10015.0
- 0_10030.0
- ...
